In [1]:
import os
# Disable ALL progress bars
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
logging.set_verbosity_error()
import logging
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error() # Only show actual errors, hide warnings

In [3]:

import warnings
warnings.filterwarnings('ignore')  # Suppress unnecessary warnings
import torch
# Choose the Qwen 2.5 1.5B Instruct model
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Load model with automatic dtype and device placement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()
print("Model loaded successfully.")
print("Device map:", model.hf_device_map if hasattr(model, "hf_device_map") else "Single device")

Model loaded successfully.
Device map: Single device


In [4]:
def generate_reply(chat_history, user_message, max_new_tokens=256):
    # Append the new user message to the history
    chat_history.append({"role": "user", "content": user_message})
    # Apply Qwen chat template to convert messages into a single text prompt
    prompt_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True,  # model expects a generation prompt
    )
    # Tokenize the prompt
    inputs = tokenizer(
        [prompt_text],
        return_tensors="pt"
    ).to(model.device)
    # Generate the model output
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )

    # Extract only the newly generated tokens
    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    # Decode the assistant's reply
    assistant_reply = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()
    # Append assistant reply to history
    chat_history.append({"role": "assistant", "content": assistant_reply})
    return chat_history, assistant_reply

In [8]:
def run_chatbot():
    chat_history = []
    print("Chatbot: Hello! I am your AI assistant. Ask me anything.")
    print("Type 'exit' or 'quit' to stop.\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! ")
            break

        chat_history, assistant_reply = generate_reply(chat_history, user_input)
        print("Chatbot:", assistant_reply)

In [9]:
run_chatbot()

Chatbot: Hello! I am your AI assistant. Ask me anything.
Type 'exit' or 'quit' to stop.

You: Hello
Chatbot: Hello! How can I assist you today?
You: What is Artificial Intelligence?
Chatbot: Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include learning (the acquisition of information and rules for using the information), reasoning (using rules to reach approximate or definite conclusions), and self-correction. The goal of AI is to create systems that can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. AI encompasses a variety of fields, including machine learning, natural language processing, computer vision, robotics, and more. It is used in numerous applications, from virtual assistants like Siri and Alexa to self-driving cars, medical diagnosis software, and even in areas like cybersecuri